                                                              بسم الله الرحمن الرحیم 

                                                                     موضوع: 
                                                         تبیین پذیری و تحلیل زیر ساختار ها


                                                              استاد : جناب آقای دکتر موسوی             
                                                       تدوین کننده : مریم مداح علی و پانیذ آذرنیا

                                                              درس: تحلیل شبکه های پیچیده



                                                                    مقدمه
در این بخش گزارش ، اجرای عملی پروژه گام به گام نمایش داده شده است. توضیحات کامل و تحلیل هر بخش از کدها (شامل ماژول‌های استخراج داده، ساخت گراف، آموزش مدل و استخراج مسیرها) در گزارش اصلی به صورت پی‌دی‌اف ارائه شده است. در اینجا صرفاً عملکرد کلی کدها و خروجی‌های نهایی نشان داده می‌شود تا خواننده بتواند نتایج را به صورت تعاملی مشاهده کند. برای مطالعه جزئیات فنی و تفسیر نتایج، به گزارش پی‌دی‌اف مراجعه شود.  

1 . ماژول chem_gene.py:

In [1]:
import pandas as pd

def get_chem_gene():
    path = "CTD_chem_gene_ixns.tsv.gz"

    df = pd.read_csv(
        path,
        sep="\t",
        compression="gzip",
        comment="#",
        header=None,
        low_memory=False
    )

    df.columns = [
        "ChemicalID",
        "ChemicalName",
        "CasRN",
        "GeneSymbol",
        "GeneID",
        "GeneForm",
        "Organism",
        "OrganismID",
        "Interaction",
        "InteractionActions",
        "PubMedIDs"
    ]


    df.to_csv("CTD_chem_gene_ixns.tsv.csv", index=False)
  
    df = df[df["Organism"] == "Homo sapiens"]

    df = df[df["GeneForm"] == "gene"]

    df = df[["ChemicalID", "GeneID"]].dropna()

    df = df.drop_duplicates()


    return df

if __name__ == "__main__":
    df = get_chem_gene()
    print("Dataset created successfully!")
    print("Number of rows:", len(df))
    print(df.head())


Dataset created successfully!
Number of rows: 42661
                                            ChemicalID  GeneID
21   10,11-dihydro-10,11-dihydroxy-5H-dibenzazepine...    2052
136                            10-hydroxynortriptyline    1565
142                                 10-hydroxywarfarin    1559
590                           1,1,2,2-tetrabromoethane   80045
591                           1,1,2,2-tetrabromoethane    4009


2.ماژول gene_disease.py :

In [3]:
import pandas as pd
import os

def get_gene_disease():
    path = "CTD_genes_diseases.tsv.gz"

    chunksize = 200_000
    edges = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        compression="gzip",
        comment="#",
        header=None,
        chunksize=chunksize,
        low_memory=False
    ):
        chunk.columns = [
            "GeneSymbol",
            "GeneID",
            "DiseaseName",
            "DiseaseID",
            "DirectEvidence",
            "InferenceGeneSymbol",
            "InferenceScore",
            "OmimIDs",
            "PubMedIDs"
        ]

        chunk = chunk[chunk["GeneID"].notna()]
        chunk = chunk[chunk["DirectEvidence"].notna()]


        chunk = chunk[["GeneID", "GeneSymbol", "DiseaseID", "DiseaseName"]]

        edges.append(chunk)

    df = pd.concat(edges).drop_duplicates()
    return df

if __name__ == "__main__":
    df = get_gene_disease()
    print("Gene-Disease dataset created successfully!")
    print("Number of edges:", len(df))
    print(df.head())

import pandas as pd
import os

def get_gene_disease():
    path = "CTD_genes_diseases.tsv.gz"

    chunksize = 200_000
    edges = []

    for chunk in pd.read_csv(
        path,
        sep="\t",
        compression="gzip",
        comment="#",
        header=None,
        chunksize=chunksize,
        low_memory=False
    ):
        chunk.columns = [
            "GeneSymbol",
            "GeneID",
            "DiseaseName",
            "DiseaseID",
            "DirectEvidence",
            "InferenceGeneSymbol",
            "InferenceScore",
            "OmimIDs",
            "PubMedIDs"
        ]

        chunk = chunk[chunk["GeneID"].notna()]
        chunk = chunk[chunk["DirectEvidence"].notna()]


        chunk = chunk[["GeneID", "GeneSymbol", "DiseaseID", "DiseaseName"]]

        edges.append(chunk)

    df = pd.concat(edges).drop_duplicates()
    return df

if __name__ == "__main__":
    df = get_gene_disease()
    print("Gene-Disease dataset created successfully!")
    print("Number of edges:", len(df))
    print(df.head())

output_path = os.path.join(os.getcwd(), "gene_disease_edges.csv")
df.to_csv(output_path, index=False)
print("File saved to:", output_path)

Gene-Disease dataset created successfully!
Number of edges: 34247
       GeneID GeneSymbol     DiseaseID                DiseaseName
14171   50518          A  MESH:D003872                 Dermatitis
14190   50518          A  MESH:D003920          Diabetes Mellitus
14201   50518          A  MESH:D003924  Diabetes Mellitus, Type 2
14217   50518          A  MESH:D003928     Diabetic Nephropathies
14278   50518          A  MESH:D004487                      Edema
Gene-Disease dataset created successfully!
Number of edges: 34247
       GeneID GeneSymbol     DiseaseID                DiseaseName
14171   50518          A  MESH:D003872                 Dermatitis
14190   50518          A  MESH:D003920          Diabetes Mellitus
14201   50518          A  MESH:D003924  Diabetes Mellitus, Type 2
14217   50518          A  MESH:D003928     Diabetic Nephropathies
14278   50518          A  MESH:D004487                      Edema
File saved to: d:\phyton\dataset\gene_disease_edges.csv


3.ماژول merge_data.py:

In [4]:
from chem_gene import *
from gene_disease import *

import pandas as pd
from collections import defaultdict, deque

CANCER_ROOT = "0004992"


def merge():

    ceg = get_chem_gene()
    cgd = get_gene_disease()

    ceg["GeneID"] = ceg["GeneID"].astype(str)
    cgd["GeneID"] = cgd["GeneID"].astype(str)

    ceps_raw = ceg.merge(cgd, on="GeneID", how="inner")
    print("After Gene merge:", ceps_raw.shape)

    if ceps_raw.empty:
        print("WARNING: CEPS raw is empty after Gene merge")
        return ceps_raw

    mask = ceps_raw["DiseaseID"].astype(str).str.match(r"^(MESH:|OMIM:)")
    ceps_raw = ceps_raw[mask].copy()
    ceps_raw["DiseaseID_norm"] = ceps_raw["DiseaseID"]


    mondo_xref = pd.read_csv("mondo/mondo_references.csv", dtype=str)

    mondo_xref = mondo_xref[
        mondo_xref["ontology"].isin(["OMIM", "MESH"])
    ][["ontology", "ontology_id", "mondo_id"]]

    mondo_xref["DiseaseID_norm"] = (
        mondo_xref["ontology"] + ":" + mondo_xref["ontology_id"]
    )

    mondo_xref = mondo_xref[["DiseaseID_norm", "mondo_id"]]
    mondo_xref.columns = ["DiseaseID_norm", "MONDO"]


    ceps = ceps_raw.merge(
        mondo_xref,
        on="DiseaseID_norm",
        how="left"
    )

    print("Before MONDO drop:", ceps.shape)
    print("Mapped MONDO count:", ceps["MONDO"].notna().sum())

    unmapped = ceps[ceps["MONDO"].isna()][["DiseaseID"]].drop_duplicates()
    if not unmapped.empty:
        unmapped.to_csv("unmapped_diseases.csv", index=False)
        print(f"Unmapped diseases saved: {len(unmapped)}")

    ceps = ceps.dropna(subset=["MONDO"])
    if ceps.empty:
        print("WARNING: All rows dropped after MONDO mapping")
        return ceps

    ceps["MONDO"] = ceps["MONDO"].str.replace(r"^MONDO:", "", regex=True)


    parents = pd.read_csv("mondo/mondo_parents.csv", dtype=str)

    parents = parents[
        parents["parent"].str.fullmatch(r"\d+") &
        parents["child"].str.fullmatch(r"\d+")
    ]

    graph = defaultdict(list)
    for _, row in parents.iterrows():
        graph[row["parent"]].append(row["child"])

    print("Children of cancer root:", graph.get(CANCER_ROOT, [])[:10])

    def get_descendants(root):
        seen = set()
        q = deque([root])
        while q:
            node = q.popleft()
            for child in graph.get(node, []):
                if child not in seen:
                    seen.add(child)
                    q.append(child)
        return seen

    cancer_mondos = get_descendants(CANCER_ROOT)
    cancer_mondos.add(CANCER_ROOT)

    print("Cancer set size:", len(cancer_mondos))
    print("Unique MONDOs before filter:", ceps["MONDO"].nunique())

    ceps = ceps[ceps["MONDO"].isin(cancer_mondos)]

    print("Final CEPS shape after filter:", ceps.shape)
    print("Unique cancer MONDOs:", ceps["MONDO"].nunique())

    if not ceps.empty:
        print("Sample Disease Names:")
        print(ceps["DiseaseName"].unique()[:20])
        ceps.to_csv("merge.csv", index=False)
    else:
        print("WARNING: Dataset empty after cancer filter")

    return ceps


if __name__ == "__main__":
    df = merge()
    print("Merge completed.")
    print("Final dataset size:", df.shape)

After Gene merge: (142254, 5)
Before MONDO drop: (146848, 7)
Mapped MONDO count: 106274
Unmapped diseases saved: 932
Children of cancer root: ['0000376', '0000621', '0000637', '0000653', '0002087', '0002100', '0002149', '0002402', '0002516', '0002813']
Cancer set size: 2293
Unique MONDOs before filter: 4528
Final CEPS shape after filter: (16764, 7)
Unique cancer MONDOs: 134
Sample Disease Names:
<StringArray>
[                     'Carcinoma',      'Carcinoma, Hepatocellular',
       'Leukemia, Myeloid, Acute',          'Lymphoma, Non-Hodgkin',
               'Multiple Myeloma',            'Prostatic Neoplasms',
                'Brain Neoplasms',                 'Adenocarcinoma',
       'Carcinoma, Squamous Cell', 'Leukemia, Promyelocytic, Acute',
              'Ovarian Neoplasms',                   'Blast Crisis',
       'Adrenocortical Carcinoma',               'Kidney Neoplasms',
    'Lymphoma, T-Cell, Cutaneous',           'Mandibular Neoplasms',
                'Sezary Syndrome', 

4. ماژول  hetero_graph.py :

In [6]:
import torch
from torch_geometric.data import HeteroData
from merge_data import merge

def graph_hetero():
    ceps_df = merge()

    if ceps_df.empty:
        print("WARNING: Input dataset is empty. Graph not created.")
        return None

    print("Input dataset shape:", ceps_df.shape)

    ceps_df["ChemicalID"] = ceps_df["ChemicalID"].astype(str)
    ceps_df["GeneID"] = ceps_df["GeneID"].astype(str)
    ceps_df["MONDO"] = ceps_df["MONDO"].astype(int)

    chemical_ids = ceps_df["ChemicalID"].unique()
    gene_ids = ceps_df["GeneID"].unique()
    disease_ids = ceps_df["MONDO"].unique()

    chemical_id_to_name = {c: c for c in chemical_ids}


    if 'GeneSymbol' in ceps_df.columns:
        gene_id_to_name = ceps_df.groupby('GeneID')['GeneSymbol'].first().to_dict()
    else:
        gene_id_to_name = {g: g for g in gene_ids}

    if 'DiseaseName' in ceps_df.columns:
        disease_id_to_name = ceps_df.groupby('MONDO')['DiseaseName'].first().to_dict()
    else:
        disease_id_to_name = {d: str(d) for d in disease_ids}

    chemical2idx = {c: i for i, c in enumerate(chemical_ids)}
    gene2idx = {g: i for i, g in enumerate(gene_ids)}
    disease2idx = {d: i for i, d in enumerate(disease_ids)}

    chem_gene_src = torch.tensor(
        ceps_df["ChemicalID"].map(chemical2idx).values, dtype=torch.long
    )
    chem_gene_dst = torch.tensor(
        ceps_df["GeneID"].map(gene2idx).values, dtype=torch.long
    )

    gene_disease_src = torch.tensor(
        ceps_df["GeneID"].map(gene2idx).values, dtype=torch.long
    )
    gene_disease_dst = torch.tensor(
        ceps_df["MONDO"].map(disease2idx).values, dtype=torch.long
    )

    data = HeteroData()
    data["chemical"].num_nodes = len(chemical_ids)
    data["gene"].num_nodes = len(gene_ids)
    data["disease"].num_nodes = len(disease_ids)

    data[("chemical", "chemical_gene", "gene")].edge_index = torch.stack(
        [chem_gene_src, chem_gene_dst]
    )
    data[("gene", "gene_disease", "disease")].edge_index = torch.stack(
        [gene_disease_src, gene_disease_dst]
    )


    data.idx2chemical = {i: chemical_id_to_name[chem_id] for i, chem_id in enumerate(chemical_ids)}
    data.idx2gene = {i: gene_id_to_name[gene_id] for i, gene_id in enumerate(gene_ids)}
    data.idx2disease = {i: disease_id_to_name[disease_id] for i, disease_id in enumerate(disease_ids)}


    data_homo = data.to_homogeneous()

    hom_idx2name = {}
    for node_type_str, start_idx in [("chemical", 0),
                                     ("gene", data["chemical"].num_nodes),
                                     ("disease", data["chemical"].num_nodes + data["gene"].num_nodes)]:
        n_nodes = data[node_type_str].num_nodes
        for i in range(n_nodes):
            hom_idx = start_idx + i
            if node_type_str == "chemical":
                hom_idx2name[hom_idx] = f"{data.idx2chemical[i]} (Chemical)"
            elif node_type_str == "gene":
                hom_idx2name[hom_idx] = f"{data.idx2gene[i]} (Gene)"
            elif node_type_str == "disease":
                hom_idx2name[hom_idx] = f"{data.idx2disease[i]} (Disease)"

    data_homo.hom_idx2name = hom_idx2name
    data_homo.offsets = {
        "chemical": 0,
        "gene": data["chemical"].num_nodes,
        "disease": data["chemical"].num_nodes + data["gene"].num_nodes
    }

    if data_homo.num_nodes > 0 and data_homo.num_edges > 0:
        print("Graph successfully created.")
        print("Total nodes:", data_homo.num_nodes)
        print("Total edges:", data_homo.num_edges)
        print("Edge types:", data_homo.edge_type.unique())
    else:
        print("Graph creation failed.")

    return data_homo

if __name__ == "__main__":
    data_homo = graph_hetero()
    if data_homo is not None:
        torch.save(data_homo, "hetero_graph.pt")
        print("Graph + full metadata saved successfully")

After Gene merge: (142254, 5)
Before MONDO drop: (146848, 7)
Mapped MONDO count: 106274
Unmapped diseases saved: 932
Children of cancer root: ['0000376', '0000621', '0000637', '0000653', '0002087', '0002100', '0002149', '0002402', '0002516', '0002813']
Cancer set size: 2293
Unique MONDOs before filter: 4528
Final CEPS shape after filter: (16764, 7)
Unique cancer MONDOs: 134
Sample Disease Names:
<StringArray>
[                     'Carcinoma',      'Carcinoma, Hepatocellular',
       'Leukemia, Myeloid, Acute',          'Lymphoma, Non-Hodgkin',
               'Multiple Myeloma',            'Prostatic Neoplasms',
                'Brain Neoplasms',                 'Adenocarcinoma',
       'Carcinoma, Squamous Cell', 'Leukemia, Promyelocytic, Acute',
              'Ovarian Neoplasms',                   'Blast Crisis',
       'Adrenocortical Carcinoma',               'Kidney Neoplasms',
    'Lymphoma, T-Cell, Cutaneous',           'Mandibular Neoplasms',
                'Sezary Syndrome', 

5. ماژول RGCN.py :

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import json
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

graph_path = "hetero_graph.pt"

try:
    data = torch.load(graph_path).to(device)
    print(f"[INFO] Graph loaded successfully: {data.num_nodes} nodes, {data.num_edges} edges")
    print(f"[INFO] Edge types: {data.edge_type.unique()}")
except FileNotFoundError:
    raise FileNotFoundError(
        f"{graph_path} not found. Run hetero_graph.py first to build and save the graph."
    )

embedding_dim = 256
hidden_dim = 256
out_dim = 64
rel_embedding_dim = 64
lr = 0.001
weight_decay = 1e-3
epochs = 100
weight_positive = 5.0
early_stopping_patience = 20
warmup_epochs = 10
dropout = 0.6

num_relations = int(data.edge_type.max()) + 1

node_emb = nn.Embedding(data.num_nodes, embedding_dim).to(device)
x = node_emb.weight

class RGCNEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, num_relations, dropout=0.6):
        super().__init__()
        self.conv1 = RGCNConv(in_dim, hidden_dim, num_relations)
        self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations)
        self.conv3 = RGCNConv(hidden_dim, out_dim, num_relations)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_type):
        x = F.relu(self.conv1(x, edge_index, edge_type))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index, edge_type))
        x = self.dropout(x)
        x = self.conv3(x, edge_index, edge_type)
        return x

class DistMultDecoder(nn.Module):
    def __init__(self, num_relations, node_emb_dim, rel_emb_dim):
        super().__init__()
        self.relation_emb = nn.Embedding(num_relations, rel_emb_dim)
        self.proj = nn.Linear(node_emb_dim, rel_emb_dim)

    def forward(self, z, edge_index, edge_type):
        head, tail = edge_index[0], edge_index[1]
        h = self.proj(z[head])
        t = self.proj(z[tail])
        r = self.relation_emb(edge_type)
        return (h * r * t).sum(dim=-1)

encoder = RGCNEncoder(embedding_dim, hidden_dim, out_dim, num_relations, dropout).to(device)
decoder = DistMultDecoder(num_relations, out_dim, rel_embedding_dim).to(device)

all_edges = data.edge_index.cpu().numpy().T          
all_types = data.edge_type.cpu().numpy()           

train_edges_list = []
val_edges_list = []
train_types_list = []
val_types_list = []

for rel in range(num_relations):
    mask = all_types == rel
    rel_edges = all_edges[mask]
    if len(rel_edges) == 0:
        continue
    train, val = train_test_split(rel_edges, test_size=0.2, random_state=42)
    train_edges_list.append(train)
    val_edges_list.append(val)
    train_types_list.extend([rel] * len(train))
    val_types_list.extend([rel] * len(val))

train_edges = torch.tensor(np.concatenate(train_edges_list, axis=0), dtype=torch.long).t().to(device)
val_edges = torch.tensor(np.concatenate(val_edges_list, axis=0), dtype=torch.long).t().to(device)
train_types = torch.tensor(train_types_list, dtype=torch.long).to(device)
val_types = torch.tensor(val_types_list, dtype=torch.long).to(device)

edge_index_train = train_edges
edge_type_train = train_types

print(f"[INFO] Train edges: {train_edges.size(1)}")
print(f"[INFO] Val edges: {val_edges.size(1)}")

def relation_aware_negative_sampling(pos_edges, pos_types, num_nodes, num_neg_per_pos=3):

    pos_edges = pos_edges.t()  
    E = pos_edges.size(0)
    total_neg = E * num_neg_per_pos

    neg_edges = torch.zeros(total_neg, 2, dtype=torch.long, device=pos_edges.device)
    neg_types = torch.zeros(total_neg, dtype=torch.long, device=pos_types.device)

    for i in range(num_neg_per_pos):
        start = i * E
        end = (i + 1) * E
        neg_edges[start:end] = pos_edges.clone()
        neg_types[start:end] = pos_types

        corrupt_head = torch.randint(0, 2, (E,), device=pos_edges.device).bool()

        random_nodes = torch.randint(0, num_nodes, (E,), device=pos_edges.device)
 
        neg_edges[start:end, 0] = torch.where(corrupt_head, random_nodes, pos_edges[:, 0])
        neg_edges[start:end, 1] = torch.where(~corrupt_head, random_nodes, pos_edges[:, 1])

    return neg_edges.t().contiguous(), neg_types

optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()) + list(node_emb.parameters()),
    lr=lr, weight_decay=weight_decay
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, verbose=True)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight_positive, device=device))

metrics = {"train_loss": [], "val_loss": [], "val_auc": []}
best_val_auc = 0
patience_counter = 0

print("\n[INFO] Starting training with relation-aware negative sampling...\n")

for epoch in range(1, epochs + 1):
    encoder.train()
    decoder.train()
    optimizer.zero_grad()


    z = encoder(x, edge_index_train, edge_type_train)

    neg_edges, neg_types = relation_aware_negative_sampling(
        train_edges, train_types, data.num_nodes, num_neg_per_pos=3
    )


    pos_scores = decoder(z, train_edges, train_types)
    neg_scores = decoder(z, neg_edges, neg_types)

    
    y = torch.cat([
        torch.ones(pos_scores.size(0), device=device),
        torch.zeros(neg_scores.size(0), device=device)
    ])
    loss = criterion(torch.cat([pos_scores, neg_scores]), y)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(encoder.parameters()) + list(decoder.parameters()) + list(node_emb.parameters()),
        max_norm=5.0
    )
    optimizer.step()

    
    if epoch == warmup_epochs:
        node_emb.requires_grad_(False)
        
        optimizer = torch.optim.Adam(
            list(encoder.parameters()) + list(decoder.parameters()),
            lr=lr, weight_decay=weight_decay
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=10, verbose=True
        )
        print(f"[INFO] Node embeddings frozen after epoch {warmup_epochs}. Optimizer recreated.")

    
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        
        neg_val_edges, neg_val_types = relation_aware_negative_sampling(
            val_edges, val_types, data.num_nodes, num_neg_per_pos=3
        )

        val_pos_scores = decoder(z, val_edges, val_types)
        val_neg_scores = decoder(z, neg_val_edges, neg_val_types)

        val_y = torch.cat([
            torch.ones(val_pos_scores.size(0)),
            torch.zeros(val_neg_scores.size(0))
        ]).cpu().numpy()
        val_scores = torch.sigmoid(torch.cat([val_pos_scores, val_neg_scores])).cpu().numpy()
        val_loss = criterion(
            torch.cat([val_pos_scores, val_neg_scores]),
            torch.tensor(val_y, device=device)
        ).item()
        val_auc = roc_auc_score(val_y, val_scores)

    metrics["train_loss"].append(loss.item())
    metrics["val_loss"].append(val_loss)
    metrics["val_auc"].append(val_auc)

    scheduler.step(val_auc)

    print(f"Epoch {epoch:03d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0

        torch.save({
            "encoder_state_dict": encoder.state_dict(),
            "decoder_state_dict": decoder.state_dict(),
            "node_emb_state_dict": node_emb.state_dict(),
            "embedding_dim": embedding_dim,
            "hidden_dim": hidden_dim,
            "out_dim": out_dim,
            "rel_embedding_dim": rel_embedding_dim,
            "num_relations": num_relations,
            "decoder_type": "DistMult"
        }, "rgcn_distmult_best.pt")

        print("[INFO] Best model saved.")
    else:
        patience_counter += 1
        if patience_counter >= early_stopping_patience:
            print(f"[INFO] Early stopping at epoch {epoch}")
            break

with open("rgcn_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\nTraining complete.")
print(f"Best Val AUC: {best_val_auc:.4f}")

[INFO] Using device: cuda


C:\Users\lotus\AppData\Local\Temp\ipykernel_21524\2369455423.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(graph_path).to(device)


[INFO] Graph loaded successfully: 2670 nodes, 33528 edges
[INFO] Edge types: tensor([0, 1], device='cuda:0')
[INFO] Train edges: 26822
[INFO] Val edges: 6706

[INFO] Starting training with relation-aware negative sampling...



d:\phyton\dataset\pytorch_env2\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 001 | Train Loss: 7.0828 | Val Loss: 6.5709 | Val AUC: 0.5586
[INFO] Best model saved.
Epoch 002 | Train Loss: 5.3029 | Val Loss: 5.1664 | Val AUC: 0.6363
[INFO] Best model saved.
Epoch 003 | Train Loss: 4.4403 | Val Loss: 4.3514 | Val AUC: 0.6228
Epoch 004 | Train Loss: 3.6434 | Val Loss: 3.6655 | Val AUC: 0.6686
[INFO] Best model saved.
Epoch 005 | Train Loss: 3.2641 | Val Loss: 3.2340 | Val AUC: 0.6469
Epoch 006 | Train Loss: 2.9928 | Val Loss: 3.0191 | Val AUC: 0.6717
[INFO] Best model saved.
Epoch 007 | Train Loss: 2.2843 | Val Loss: 2.4010 | Val AUC: 0.7193
[INFO] Best model saved.
Epoch 008 | Train Loss: 2.0749 | Val Loss: 2.1422 | Val AUC: 0.7353
[INFO] Best model saved.
Epoch 009 | Train Loss: 1.9014 | Val Loss: 2.0005 | Val AUC: 0.7006
[INFO] Node embeddings frozen after epoch 10. Optimizer recreated.
Epoch 010 | Train Loss: 1.6000 | Val Loss: 1.6628 | Val AUC: 0.7219
Epoch 011 | Train Loss: 1.4187 | Val Loss: 1.4555 | Val AUC: 0.7154
Epoch 012 | Train Loss: 1.4139 | Va

d:\phyton\dataset\pytorch_env2\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 013 | Train Loss: 1.2071 | Val Loss: 1.2756 | Val AUC: 0.7537
Epoch 014 | Train Loss: 1.2428 | Val Loss: 1.2630 | Val AUC: 0.7328
Epoch 015 | Train Loss: 1.2027 | Val Loss: 1.2111 | Val AUC: 0.7345
Epoch 016 | Train Loss: 1.1193 | Val Loss: 1.1800 | Val AUC: 0.7476
Epoch 017 | Train Loss: 1.0466 | Val Loss: 1.0856 | Val AUC: 0.7847
[INFO] Best model saved.
Epoch 018 | Train Loss: 1.0048 | Val Loss: 1.0975 | Val AUC: 0.7750
Epoch 019 | Train Loss: 0.9885 | Val Loss: 1.0592 | Val AUC: 0.8004
[INFO] Best model saved.
Epoch 020 | Train Loss: 0.9534 | Val Loss: 1.0281 | Val AUC: 0.7985
Epoch 021 | Train Loss: 0.9499 | Val Loss: 1.0145 | Val AUC: 0.7967
Epoch 022 | Train Loss: 0.9110 | Val Loss: 0.9821 | Val AUC: 0.8129
[INFO] Best model saved.
Epoch 023 | Train Loss: 0.9089 | Val Loss: 0.9935 | Val AUC: 0.8068
Epoch 024 | Train Loss: 0.8704 | Val Loss: 0.9399 | Val AUC: 0.8249
[INFO] Best model saved.
Epoch 025 | Train Loss: 0.8736 | Val Loss: 0.9469 | Val AUC: 0.8198
Epoch 026 | Trai

6. ماژول GNN_Explainer.py :

In [8]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] GNN Explainer running on: {device}")



def explain_link_edge_aware(
    z,
    edge_index,
    src,
    dst,
    link_score,
    epochs=300,
    lr=0.1,
    lam_sparse=0.01,
    lam_entropy=0.01,
    verbose=True
):


    device = z.device
    num_edges = edge_index.size(1)

    edge_mask_param = nn.Parameter(
        torch.randn(num_edges, device=device)
    )

    optimizer = torch.optim.Adam([edge_mask_param], lr=lr)

    with torch.no_grad():
        base_score = link_score(src, dst)

    if verbose:
        print(f"[INFO] Base link score: {base_score:.4f}")

    for epoch in range(epochs):
        mask = torch.sigmoid(edge_mask_param)


        node_weight = torch.zeros(z.size(0), device=device)
        node_weight.index_add_(0, edge_index[0], mask)
        node_weight.index_add_(0, edge_index[1], mask)

        node_weight = node_weight / (node_weight.max() + 1e-8)
        z_masked = z * node_weight.unsqueeze(1)

        pred = link_score(src, dst, z_override=z_masked, return_tensor=True)

        loss_pred = (pred - base_score) ** 2
        loss_sparse = mask.mean()
        loss_entropy = (-mask * torch.log(mask + 1e-8)).mean()

        loss = loss_pred + lam_sparse * loss_sparse + lam_entropy * loss_entropy

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if verbose and epoch % 50 == 0:
            print(f"[Epoch {epoch:03d}] loss={loss.item():.4f} pred={pred.item():.4f}")

    return torch.sigmoid(edge_mask_param).detach()



def important_nodes_from_edge_mask(edge_mask, edge_index, ratio=0.4):

    num_nodes = int(edge_index.max()) + 1
    node_score = torch.zeros(num_nodes, device=edge_mask.device)

    node_score.index_add_(0, edge_index[0], edge_mask)
    node_score.index_add_(0, edge_index[1], edge_mask)

    k = max(1, int(num_nodes * ratio))
    return set(node_score.topk(k).indices.tolist())


def enumerate_ceps_paths_masked(
    important_nodes,
    adj,
    chemical_nodes,
    gene_nodes,
    disease_nodes,
    link_score
):
    paths = []

    for c in chemical_nodes:
        if c not in important_nodes:
            continue

        for g in gene_nodes:
            if g not in important_nodes or g not in adj[c]:
                continue

            for d in disease_nodes:
                if d not in important_nodes or d not in adj[g]:
                    continue

                score = link_score(c, g) + link_score(g, d)
                paths.append((score, [c, g, d]))

    paths.sort(reverse=True, key=lambda x: x[0])
    return paths


def explain_and_enumerate_paths(
    z,
    edge_index,
    src,
    dst,
    adj,
    chemical_nodes,
    gene_nodes,
    disease_nodes,
    link_score,
    node_ratio=0.4,
    epochs=300
):
    print("[INFO] Running GNN Explainer (edge‑aware)...")

    edge_mask = explain_link_edge_aware(
        z=z,
        edge_index=edge_index,
        src=src,
        dst=dst,
        link_score=link_score,
        epochs=epochs
    )

    important_nodes = important_nodes_from_edge_mask(
        edge_mask, edge_index, ratio=node_ratio
    )

    print(f"[INFO] Important nodes selected: {len(important_nodes)}")

    print("[INFO] Enumerating CEPS-style paths...")
    paths = enumerate_ceps_paths_masked(
        important_nodes,
        adj,
        chemical_nodes,
        gene_nodes,
        disease_nodes,
        link_score
    )

    print(f"[INFO] Total paths found: {len(paths)}")

    return {
        "edge_mask": edge_mask,
        "important_nodes": important_nodes,
        "paths": paths
    }

[INFO] GNN Explainer running on: cuda


7. ماژول CEPS.py :

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
import networkx as nx
import plotly.graph_objects as go
import webbrowser


data = None
z = None
edge_index = None
edge_type = None
adj = None

chemical_nodes = None
gene_nodes = None
disease_nodes = None
offsets = None

all_paths = None
paths_by_pair = None

def initialize():

    global data, z, edge_index, edge_type, adj
    global chemical_nodes, gene_nodes, disease_nodes
    global offsets

    if z is not None:

        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] Initializing CEPS on: {device}")


    data = torch.load(
        "hetero_graph.pt",
        map_location=device,
        weights_only=False
    ).to(device)

    num_nodes = data.num_nodes
    edge_index = data.edge_index
    edge_type = data.edge_type
    offsets = data.offsets

    chemical_nodes = list(range(offsets["chemical"], offsets["gene"]))
    gene_nodes     = list(range(offsets["gene"], offsets["disease"]))
    disease_nodes  = list(range(offsets["disease"], num_nodes))

    class RGCNEncoder(nn.Module):
        def __init__(self, in_dim, hidden_dim, out_dim, num_relations, dropout=0.6):
            super().__init__()
            self.conv1 = RGCNConv(in_dim, hidden_dim, num_relations)
            self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations)
            self.conv3 = RGCNConv(hidden_dim, out_dim, num_relations)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x, edge_index, edge_type):
            x = F.relu(self.conv1(x, edge_index, edge_type))
            x = self.dropout(x)
            x = F.relu(self.conv2(x, edge_index, edge_type))
            x = self.dropout(x)
            x = self.conv3(x, edge_index, edge_type)
            return x

    checkpoint = torch.load(
        "rgcn_distmult_best.pt",
        map_location=device,
        weights_only=False
    )

    embedding_dim = checkpoint["embedding_dim"]
    hidden_dim = checkpoint["hidden_dim"]
    out_dim = checkpoint["out_dim"]
    num_relations = checkpoint["num_relations"]

    node_emb = nn.Embedding(num_nodes, embedding_dim).to(device)
    node_emb.load_state_dict(checkpoint["node_emb_state_dict"])

    encoder = RGCNEncoder(
        embedding_dim, hidden_dim, out_dim, num_relations
    ).to(device)
    encoder.load_state_dict(checkpoint["encoder_state_dict"])
    encoder.eval()

    with torch.no_grad():
        z = encoder(node_emb.weight, edge_index, edge_type).detach()

    adj = {n: set() for n in range(num_nodes)}
    for u, v in edge_index.t().tolist():
        adj[u].add(v)
        adj[v].add(u)

    print("[INFO] CEPS initialized ✅")



def link_score(u, v, z_override=None, return_tensor=False):

    emb = z_override if z_override is not None else z

    sim = torch.cosine_similarity(
        emb[u].unsqueeze(0),
        emb[v].unsqueeze(0)
    )
    score = (sim + 1) / 2  

    if return_tensor:
        return score.squeeze()
    else:
        return score.item()

def visualize_paths(all_paths, paths_by_pair, filename="explained_paths_interactive.html"):

    initialize()

    G = nx.Graph()

    global_top_paths = all_paths
    for score, path in global_top_paths:
        c, g, d = path
        G.add_node(c, type="chemical", label=data.idx2chemical[c - offsets["chemical"]])
        G.add_node(g, type="gene", label=data.idx2gene[g - offsets["gene"]])
        G.add_node(d, type="disease", label=data.idx2disease[d - offsets["disease"]])


    nodes_by_type = {"chemical": [], "gene": [], "disease": []}
    for n, d in G.nodes(data=True):
        nodes_by_type[d["type"]].append(n)

    max_nodes = max(len(nodes_by_type["chemical"]),
                    len(nodes_by_type["gene"]),
                    len(nodes_by_type["disease"]))
    y_gap = max(1500, 30000 // max_nodes if max_nodes > 0 else 1500)
    x_shift = {"chemical": 0, "gene": 120, "disease": 240}

    pos = {}
    for i, n in enumerate(nodes_by_type["chemical"]):
        pos[n] = (x_shift["chemical"], -i * y_gap)
    for i, n in enumerate(nodes_by_type["gene"]):
        pos[n] = (x_shift["gene"], -i * y_gap)
    for i, n in enumerate(nodes_by_type["disease"]):
        pos[n] = (x_shift["disease"], -i * y_gap)

    edge_traces_global = []
    for global_rank, (score, path) in enumerate(global_top_paths):
        c, g, d = path
        color = "red" if global_rank < 3 else "rgba(180,180,180,0.25)"
        text_color = "darkred" if global_rank < 3 else "gray"
        for u, v in [(c, g), (g, d)]:
            x0, y0 = pos[u];
            x1, y1 = pos[v]
            xm = (x0 + x1) / 2;
            ym = (y0 + y1) / 2
            edge_traces_global.append(
                go.Scatter(x=[x0, x1], y=[y0, y1], mode="lines",
                           line=dict(width=0.8 + 3 * score, color=color),
                           hoverinfo="skip", showlegend=False)
            )
            edge_traces_global.append(
                go.Scatter(x=[xm], y=[ym + 20], mode="text", text=[f"{score:.3f}"],
                           textfont=dict(size=12, color=text_color),
                           showlegend=False, hoverinfo="skip")
            )

    edge_traces_local = []
    trace_meta_local = []
    TOP_K_PER_PAIR = 5

    for (c, d), paths in paths_by_pair.items():
        if c not in G.nodes() or d not in G.nodes():
            continue
        for score, path, global_rank, local_rank in paths[:TOP_K_PER_PAIR]:
            _, g, _ = path
            if g not in G.nodes():
                continue
            color = "red" if local_rank < 3 else "rgba(180,180,180,0.25)"
            text_color = "darkred" if local_rank < 3 else "gray"
            for u, v in [(c, g), (g, d)]:
                x0, y0 = pos[u];
                x1, y1 = pos[v]
                xm = (x0 + x1) / 2;
                ym = (y0 + y1) / 2
                edge_traces_local.append(
                    go.Scatter(x=[x0, x1], y=[y0, y1], mode="lines",
                               line=dict(width=0.8 + 3 * score, color=color),
                               hoverinfo="skip", showlegend=False)
                )
                trace_meta_local.append((c, d))
                edge_traces_local.append(
                    go.Scatter(x=[xm], y=[ym + 20], mode="text", text=[f"{score:.3f}"],
                               textfont=dict(size=12, color=text_color),
                               showlegend=False, hoverinfo="skip")
                )
                trace_meta_local.append((c, d))

    def node_trace_for_type(node_type, color, size):
        xs, ys, texts, hovers = [], [], [], []
        for n, d in G.nodes(data=True):
            if d["type"] == node_type:
                x, y = pos[n]
                xs.append(x);
                ys.append(y)
                label = d["label"]
                hover = f"<b>{label}</b><br>Type: {node_type}<br>ID: {n}"
                texts.append(label)
                hovers.append(hover)
        return go.Scatter(
            x=xs, y=ys, mode="markers+text", text=texts,
            textposition="middle right", hoverinfo="text", hovertext=hovers,
            marker=dict(size=size, color=color, line=dict(width=1.5, color="black")),
            showlegend=True, name=node_type.capitalize()
        )

    node_traces = [
        node_trace_for_type("chemical", "#8dd3c7", 20),
        node_trace_for_type("gene", "#ffffb3", 16),
        node_trace_for_type("disease", "#fb8072", 20),
    ]

    buttons = []

    initial_visibility = [False] * len(edge_traces_global) + [False] * len(edge_traces_local) + [True] * len(
        node_traces)

    overview_visibility = [True] * len(edge_traces_global) + [False] * len(edge_traces_local) + [True] * len(
        node_traces)
    buttons.append(dict(
        label="Overview",
        method="update",
        args=[{"visible": overview_visibility}, {"title": "CEPS Explanation: Overview"}]
    ))

    valid_pairs_local = sorted(set(trace_meta_local))
    for (c, d) in valid_pairs_local:
        visibility = [False] * len(edge_traces_global) + \
                     [(meta == (c, d)) for meta in trace_meta_local] + \
                     [True] * len(node_traces)
        label = f"{data.idx2chemical[c - offsets['chemical']]}, {data.idx2disease[d - offsets['disease']]}"
        buttons.append(dict(
            label=label,
            method="update",
            args=[{"visible": visibility}, {"title": f"CEPS Explanation: {label}"}]
        ))

    fig = go.Figure(
        data=edge_traces_global + edge_traces_local + node_traces,
        layout=go.Layout(
            title=dict(text="Interactive CEPS Explanation", font=dict(size=20), y=0.98),
            hovermode="closest",
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            margin=dict(l=40, r=40, t=120, b=40),
            paper_bgcolor="white",
            plot_bgcolor="white",
            updatemenus=[dict(
                buttons=buttons,
                direction="down",
                x=0.01,
                y=1.12,
                showactive=True,
                bgcolor="white",
                bordercolor="#ccc",
                font=dict(size=11)
            )]
        )
    )

    for i in range(len(edge_traces_global) + len(edge_traces_local)):
        fig.data[i].visible = False

    fig.update_layout(autosize=True, margin=dict(l=80, r=80, t=120, b=80))

    config = {
        'responsive': True,
        'displaylogo': False,
        'modeBarButtonsToRemove': ['select2d', 'lasso2d', 'zoomIn2d', 'zoomOut2d', 'autoScale2d'],
        'modeBarButtonsToAdd': ['zoom2d', 'pan2d', 'resetScale2d', 'toimage']
    }

    html_str = fig.to_html(include_plotlyjs=True, config=config)
    css = """
    <style>
    .modebar-btn {
        font-size: 20px !important;
        width: 32px !important;
        height: 32px !important;
    }
    .modebar-btn .icon {
        width: 20px !important;
        height: 20px !important;
    }
    </style>
    """
    html_str = html_str.replace('</head>', css + '</head>')

    with open(filename, "w", encoding="utf-8") as f:
        f.write(html_str)

    webbrowser.open(filename)

def rank_pairs(paths_by_pair, mode="best"):

    pair_scores = []

    for (c, d), paths in paths_by_pair.items():
        scores = [score for score, *_ in paths]

        if mode == "best":
            s = max(scores)
        elif mode == "mean":
            s = sum(scores) / len(scores)
        elif mode == "sum":
            s = sum(scores)
        else:
            raise ValueError("Unknown mode")

        pair_scores.append((s, (c, d)))

    pair_scores.sort(reverse=True, key=lambda x: x[0])
    return pair_scores



if __name__ == "__main__":
    initialize()

[INFO] Initializing CEPS on: cuda
[INFO] CEPS initialized ✅


8. ماژول index_lookup.py :

In [10]:
import CEPS

def initialize_if_needed():
    if CEPS.z is None:
        CEPS.initialize()

def get_chemical_index(chemical_name):

    initialize_if_needed()

    for local_idx, name in CEPS.data.idx2chemical.items():
        if name == chemical_name:
            return local_idx + CEPS.offsets["chemical"]

    raise ValueError(f"Chemical '{chemical_name}' not found")


def get_disease_index(disease_name):

    initialize_if_needed()

    for local_idx, name in CEPS.data.idx2disease.items():
        if name == disease_name:
            return local_idx + CEPS.offsets["disease"]

    raise ValueError(f"Disease '{disease_name}' not found")

9. ماژول main.py:

In [11]:
from GNN_Explainer import explain_and_enumerate_paths
import CEPS
import csv
import torch
import index_lookup



TOP_K_PAIRS = 20       
TOP_M_PATHS = 5        

SPECIFIC_CHEMICAL = "Arsenic"
SPECIFIC_DISEASE = "Osteosarcoma"



def main():

    CEPS.initialize()


    src = CEPS.chemical_nodes[0]
    dst = CEPS.disease_nodes[0]

    print(
        "[INFO] Explaining:",
        CEPS.data.idx2chemical[src - CEPS.offsets["chemical"]],
        "→",
        CEPS.data.idx2disease[dst - CEPS.offsets["disease"]],
    )

    result = explain_and_enumerate_paths(
        z=CEPS.z,
        edge_index=CEPS.edge_index,
        src=src,
        dst=dst,
        adj=CEPS.adj,
        chemical_nodes=CEPS.chemical_nodes,
        gene_nodes=CEPS.gene_nodes,
        disease_nodes=CEPS.disease_nodes,
        link_score=CEPS.link_score,
        node_ratio=0.4,
        epochs=150,
    )

    paths = result["paths"]
    print(f"[INFO] Paths found after explanation: {len(paths)}")

    torch.save(
        {
            "src": src,
            "dst": dst,
            "paths": paths,
            "important_nodes": result["important_nodes"],
        },
        "explanation_result.pt",
    )

    with open("explained_paths.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["score", "chemical", "gene", "disease"])
        for score, (c, g, d) in paths:
            writer.writerow(
                [
                    score,
                    CEPS.data.idx2chemical[c - CEPS.offsets["chemical"]],
                    CEPS.data.idx2gene[g - CEPS.offsets["gene"]],
                    CEPS.data.idx2disease[d - CEPS.offsets["disease"]],
                ]
            )

    print("[INFO] Explained paths saved ✅")


    CEPS.all_paths = paths
    CEPS.paths_by_pair = {}

    for global_rank, (score, path) in enumerate(paths):
        c, g, d = path
        pair = (c, d)
        CEPS.paths_by_pair.setdefault(pair, []).append(
            (score, path, global_rank)
        )

    for pair in CEPS.paths_by_pair:
        CEPS.paths_by_pair[pair].sort(reverse=True, key=lambda x: x[0])
        CEPS.paths_by_pair[pair] = [
            (score, path, global_rank, local_rank)
            for local_rank, (score, path, global_rank)
            in enumerate(CEPS.paths_by_pair[pair])
        ]

    print("[INFO] Explained pairs:", len(CEPS.paths_by_pair))
    print("[INFO] Explained paths:", len(paths))


    if SPECIFIC_CHEMICAL and SPECIFIC_DISEASE:
        c = index_lookup.get_chemical_index(SPECIFIC_CHEMICAL)
        d = index_lookup.get_disease_index(SPECIFIC_DISEASE)

        pair = (c, d)
        print(
            "[INFO] Visualizing specific pair:",
            CEPS.data.idx2chemical[c - CEPS.offsets["chemical"]],
            "→",
            CEPS.data.idx2disease[d - CEPS.offsets["disease"]],
        )

        if pair not in CEPS.paths_by_pair:
            print("[WARN] No explainer-valid paths for this pair.")
        else:
            pair_paths = CEPS.paths_by_pair[pair][:TOP_M_PATHS]

            pair_paths_by_pair = {pair: pair_paths}
            pair_all_paths = [(score, path) for score, path, _, _ in pair_paths]

            CEPS.visualize_paths(
                all_paths=pair_all_paths,
                paths_by_pair=pair_paths_by_pair,
                filename="pair_specific_paths.html",
            )


    ranked_pairs = CEPS.rank_pairs(CEPS.paths_by_pair, mode="best")
    selected_pairs = [pair for _, pair in ranked_pairs[:TOP_K_PAIRS]]

    filtered_paths_by_pair = {}
    filtered_all_paths = []

    for pair in selected_pairs:
        top_paths = CEPS.paths_by_pair[pair][:TOP_M_PATHS]
        filtered_paths_by_pair[pair] = top_paths
        for score, path, _, _ in top_paths:
            filtered_all_paths.append((score, path))

    print(
        f"[INFO] Visualizing {len(filtered_paths_by_pair)} pairs "
        f"({TOP_M_PATHS} paths per pair)"
    )

    CEPS.visualize_paths(
        all_paths=filtered_all_paths,
        paths_by_pair=filtered_paths_by_pair,
        filename="top_pairs_paths.html",
    )


if __name__ == "__main__":
    main()

[INFO] GNN Explainer running on: cuda
[INFO] Initializing CEPS on: cuda
[INFO] CEPS initialized ✅
[INFO] Explaining: 10,11-dihydro-10,11-dihydroxy-5H-dibenzazepine-5-carboxamide → Carcinoma
[INFO] Running GNN Explainer (edge‑aware)...
[INFO] Base link score: 0.5945
[Epoch 000] loss=0.0080 pred=0.5945
[Epoch 050] loss=0.0012 pred=0.5945
[Epoch 100] loss=0.0003 pred=0.5945
[INFO] Important nodes selected: 1068
[INFO] Enumerating CEPS-style paths...
[INFO] Total paths found: 12898
[INFO] Paths found after explanation: 12898
[INFO] Explained paths saved ✅
[INFO] Explained pairs: 5487
[INFO] Explained paths: 12898
[INFO] Visualizing specific pair: Arsenic → Osteosarcoma
[INFO] Visualizing 20 pairs (5 paths per pair)
